# 03 — Few-shot Visual Descriptions (TIP-003)

**Goal:** generate Vietnamese descriptions of skin condition images
using Qwen2.5-VL-7B-Instruct-AWQ (the same model that will serve at
runtime), and write them to `data/visual_descriptions.json` for use
as the `VISUAL_CONTEXT` block in the system prompt (Blueprint §8).

**Why Qwen self-describe (not GPT-4o):**
- Apache 2.0 cleanliness (Risk B Approach 3 decision).
- Full reproducibility — same model that serves runtime.
- Few-shot examples align with model's own perception.

**Source dataset:** Fitzpatrick17k (CC BY-NC-SA 4.0, attribution
required, no redistribution of raw images).

**Run on Colab T4** (or any machine with CUDA + ≥ 13 GB VRAM).
Determinism is required (`temperature=0`, `seed=42`); a re-run must
produce identical descriptions.

**Acceptance hooks:**

- Qwen2.5-VL-7B-Instruct-AWQ loads under 13 GB VRAM
- Per condition: `descriptions` length = `min(5, available)`
- Each description: Vietnamese, 3–5 sentences (≤200 tokens)
- `validation.diagnostic_term_violations ≤ 2`
- Same notebook re-run → identical output
- Total runtime < 30 min on T4

## Coverage caveat (escalation flagged in Completion Report)

Per TIP-001's audit, three in-scope conditions have **zero**
Fitzpatrick17k images: `atopic_dermatitis`, `fungal_infection`,
`herpes_zoster`. TIP-001A's augment-then-keep policy closed those
gaps via SCIN + DermNet NZ, but TIP-003's spec says Fitzpatrick17k
only.

This notebook implements **strict TIP-003 by default**: those three
conditions will end up with `n_descriptions = 0`. A SCIN-fallback
path is provided as a single boolean flag at the top of Cell 4:
flip `ENABLE_SCIN_FALLBACK = True` to fill the gaps from SCIN
(license CC BY 4.0, public images on GCS). Default is `False` per
TIP-003 spec.

## Cell 1 — Setup

vLLM and Pillow are the heavy installs. `requests` for image fetches.

In [ ]:
import importlib.util, subprocess, sys

def ensure(*pkgs: str) -> None:
    missing = [p for p in pkgs
               if importlib.util.find_spec(p.split('==')[0].split('[')[0]) is None]
    if missing:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

ensure('vllm', 'pillow', 'requests', 'pandas')

import csv, hashlib, json, os, random, ssl, time
import urllib.request, urllib.error
from collections import Counter, defaultdict
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path
from PIL import Image

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
RAW_DIR = REPO_ROOT / 'data' / 'raw'
DATA_DIR = REPO_ROOT / 'data'
IMAGES_DIR = RAW_DIR / 'fitz_selected'
RAW_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# Polite, identifying UA for image fetches
USER_AGENT = ('DermAssist-VN-Research/0.1 '
              '(Apache-2.0 reference impl; non-commercial)')

print('Repo root:', REPO_ROOT)
print('Raw dir:  ', RAW_DIR, '(gitignored)')

## Cell 2 — (Colab only) Mount Drive for cache persistence

Skip on local. On Colab, mount Drive so the model weights and image
cache survive runtime restarts.

```python
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/DermAssist')
RAW_DIR = PROJECT_DIR / 'data' / 'raw'
DATA_DIR = PROJECT_DIR / 'data'
IMAGES_DIR = RAW_DIR / 'fitz_selected'
for p in (RAW_DIR, IMAGES_DIR):
    p.mkdir(parents=True, exist_ok=True)
print('Using Drive:', RAW_DIR)
```

(Code-comment so the cell doesn't error outside Colab.)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_DIR = Path('/content/drive/MyDrive/DermAssist')
# RAW_DIR = PROJECT_DIR / 'data' / 'raw'
# DATA_DIR = PROJECT_DIR / 'data'
# IMAGES_DIR = RAW_DIR / 'fitz_selected'
# for p in (RAW_DIR, IMAGES_DIR):
#     p.mkdir(parents=True, exist_ok=True)
# print('Using Drive:', RAW_DIR)

## Cell 3 — Load Qwen2.5-VL-7B-Instruct-AWQ

The HF model id `Qwen/Qwen2.5-VL-7B-Instruct-AWQ` is the AWQ INT4
quantized variant — should fit in ~6 GB VRAM with room for KV cache
on T4 (16 GB total). If that exact id 404s on HF, the alternatives
to try (in order) are:

- `Qwen/Qwen2.5-VL-7B-Instruct` (full FP16; needs L4/A100, NOT T4)
- A community AWQ port (search HF for `Qwen2.5-VL-7B AWQ`)

vLLM auto-detects vision support from the model config.

In [ ]:
from vllm import LLM, SamplingParams

QWEN_MODEL = 'Qwen/Qwen2.5-VL-7B-Instruct-AWQ'

llm = LLM(
    model=QWEN_MODEL,
    quantization='awq',
    dtype='float16',
    max_model_len=4096,
    gpu_memory_utilization=0.85,
    trust_remote_code=True,
    seed=42,
)
print(f'✓ Loaded {QWEN_MODEL}')

## Cell 4 — Constants + condition mapping

The 8 in-scope conditions and their Vietnamese names. Same taxonomy
as TIP-001 / TIP-002. `ENABLE_SCIN_FALLBACK` is the one toggle for
Chủ thầu — see the coverage caveat at the top of this notebook.

In [ ]:
CONDITIONS = [
    'atopic_dermatitis', 'fungal_infection', 'herpes_zoster', 'acne',
    'contact_dermatitis', 'eczema', 'psoriasis', 'scabies',
]

CONDITION_VN = {
    'atopic_dermatitis':  'Viêm da cơ địa',
    'fungal_infection':   'Nấm da',
    'herpes_zoster':      'Zona thần kinh',
    'acne':               'Mụn trứng cá',
    'contact_dermatitis': 'Viêm da tiếp xúc & Mề đay',
    'eczema':             'Chàm',
    'psoriasis':          'Vảy nến',
    'scabies':            'Bệnh ghẻ',
}

# Same alias map as TIP-001 / TIP-002 — single source of truth duplicated
# here for notebook self-containment.
IN_SCOPE_CONDITIONS = {
    'atopic_dermatitis':  ['atopic dermatitis', 'atopic-dermatitis', 'eczema atopic',
                           'atopic'],
    'fungal_infection':   ['tinea', 'tinea corporis', 'tinea pedis', 'tinea cruris',
                           'onychomycosis', 'fungal infection', 'candidiasis',
                           'dermatophytosis', 'dermatophyte', 'ringworm'],
    'herpes_zoster':      ['herpes zoster', 'shingles', 'zoster',
                           'varicella zoster'],
    'acne':               ['acne', 'acne vulgaris'],
    'contact_dermatitis': ['contact dermatitis', 'allergic contact dermatitis',
                           'irritant contact dermatitis', 'urticaria'],
    'eczema':             ['eczema', 'dyshidrotic eczema', 'nummular eczema'],
    'psoriasis':          ['psoriasis', 'psoriasis vulgaris', 'guttate psoriasis'],
    'scabies':            ['scabies'],
}
SPECIFICITY_ORDER = [
    'atopic_dermatitis', 'herpes_zoster', 'fungal_infection',
    'contact_dermatitis', 'psoriasis', 'scabies', 'acne', 'eczema',
]
assert set(SPECIFICITY_ORDER) == set(IN_SCOPE_CONDITIONS.keys())

def map_label(label_str: str) -> str | None:
    if not label_str:
        return None
    s = label_str.lower().strip()
    for key in SPECIFICITY_ORDER:
        for alias in IN_SCOPE_CONDITIONS[key]:
            if alias in s:
                return key
    return None

# Selection knobs
N_PER_CONDITION = 5
N_CANDIDATES = 30           # try up to this many per condition; take first 5 OK
DOWNLOAD_TIMEOUT = 12       # seconds per image
MIN_IMAGE_BYTES = 10_000    # reject tiny / placeholder images

# Toggle: DEPRECATED (TIP-003A replaced this with SOURCE_POLICY).
# Kept as True to avoid breaking re-runs from cached state.
ENABLE_SCIN_FALLBACK = True

# Cell — Source policy (replaces Fitzpatrick-only logic)
# Per-condition data source policy. Priority: best-coverage, then license-clean.
# DermNet URLs may be referenced and fetched session-only, but images are
# NEVER persisted to disk (CC BY-NC-ND restriction).
SOURCE_POLICY = {
    # 5 conditions Fitzpatrick17k handles well (atlas URLs ~93% live):
    "acne":               ["fitzpatrick17k"],          # 518 in Fitz, atlas-priority
    "contact_dermatitis": ["fitzpatrick17k"],          # 693 in Fitz
    "psoriasis":          ["fitzpatrick17k"],          # 706 in Fitz
    "scabies":            ["fitzpatrick17k"],          # 339 in Fitz
    "eczema":             ["scin", "fitzpatrick17k"],  # SCIN preferred (smartphone photos)

    # 3 conditions where Fitz17k has 0 — use augmented sources:
    "fungal_infection":   ["scin"],                    # 190 in SCIN, CC BY 4.0
    "herpes_zoster":      ["scin"],                    # 83 in SCIN, DANGEROUS
    "atopic_dermatitis":  ["dermnet_session_only"],    # 35 URLs in DermNet (no disk cache)
}

# Sanity check at notebook start: every in-scope condition has at least one source
import json
scope_path = DATA_DIR / "condition_scope.json"
audit_path = DATA_DIR / "dataset_audit.json"
if not scope_path.exists():
    raise FileNotFoundError("data/condition_scope.json missing")
if not audit_path.exists():
    raise FileNotFoundError("data/dataset_audit.json missing")
scope = json.loads(scope_path.read_text())
for cond in scope["in_scope"]:
    assert cond in SOURCE_POLICY, f"Missing source policy for in-scope condition: {cond}"
print("✓ Source policy covers all in-scope conditions")

# Polite seed for selection determinism
SELECTION_SEED = 42

# vLLM determinism settings
QWEN_SAMPLING = SamplingParams(temperature=0.0, max_tokens=200, seed=42)

# Forbidden terms in descriptions (REQ — descriptions must not name diseases)
FORBIDDEN_TERMS = [
    'chẩn đoán', 'bệnh', 'atopic', 'psoriasis', 'viêm da', 'nấm',
    'zona', 'trứng cá', 'eczema', 'chàm', 'vảy nến', 'ghẻ', 'mề đay',
]

## Cell 5 — Image selection from Fitzpatrick17k

Selection algorithm:

1. Read `data/raw/fitzpatrick17k.csv` (download metadata if missing).
2. For each in-scope condition: filter rows whose `label` maps to the
   key. Prefer rows with `qc=1` (quality-checked) and Fitzpatrick
   types 3/4/5 (relevant skin tones for VN).
3. Shuffle deterministically (`SELECTION_SEED`).
4. Try-fetch up to `N_CANDIDATES` per condition; keep first 5 that
   succeed (200 OK + image bytes > `MIN_IMAGE_BYTES` + PIL-decodable).
5. Cache to `IMAGES_DIR / {condition}/{md5}.jpg` so re-runs skip
   already-downloaded files.

**Why so much defense:** Fitzpatrick17k links to dermaamin.com and
atlasdermatologico.com.br; ~90% of dermaamin URLs are dead by 2026.
Atlas is much more reliable (~93% live) but heavily under-represented
for some conditions.

In [ ]:
FITZ_URL = 'https://raw.githubusercontent.com/mattgroh/fitzpatrick17k/main/fitzpatrick17k.csv'
FITZ_LOCAL = RAW_DIR / 'fitzpatrick17k.csv'

if not FITZ_LOCAL.exists():
    print('Downloading Fitzpatrick17k metadata...')
    urllib.request.urlretrieve(FITZ_URL, FITZ_LOCAL)
print(f'Fitzpatrick17k metadata: {FITZ_LOCAL.stat().st_size:,} bytes')


def fitz_candidates_for(condition_key: str, n: int = 30) -> list[dict]:
    """Return list of {url, fitzpatrick_type, qc, label} candidates for a condition,
    sorted with priority: III–V skin tones first, then qc=1, then atlas
    domain (more reliable), then random shuffle."""
    rng = random.Random(SELECTION_SEED + hash(condition_key) % 10_000)
    rows = []
    with open(FITZ_LOCAL) as f:
        for row in csv.DictReader(f):
            if map_label(row.get('label', '')) == condition_key:
                rows.append({
                    'url': row['url'],
                    'fitzpatrick_type': row.get('fitzpatrick_scale', ''),
                    'qc': row.get('qc', ''),
                    'label': row['label'],
                    'source_dataset': 'Fitzpatrick17k',
                    'license': 'CC BY-NC-SA 4.0',
                    'case_id': None,
                    'monk_tone': None,
                })
    # Stable sort with priority key
    def priority(r):
        atlas = 'atlas' in r['url'].lower()  # atlas more reliable
        in_band = r['fitzpatrick_type'] in ('3', '4', '5')
        # Lower numbers = higher priority
        return (
            0 if in_band else 1,
            0 if atlas else 1,
            0 if str(r.get('qc')).strip() == '1' else 1,
            rng.random(),
        )
    rows.sort(key=priority)
    return rows[:n]


# Cell — SCIN candidate fetching
import pandas as pd

SCIN_CASES_URL = "https://storage.googleapis.com/dx-scin-public-data/dataset/scin_cases.csv"
SCIN_LABELS_URL = "https://storage.googleapis.com/dx-scin-public-data/dataset/scin_labels.csv"
SCIN_IMAGE_BASE = "https://storage.googleapis.com/dx-scin-public-data/dataset/images/"

def load_scin_metadata():
    """Cached SCIN metadata. Re-uses TIP-001A download if present."""
    cache = RAW_DIR / "scin_labels.csv"
    if cache.exists():
        labels = pd.read_csv(cache)
    else:
        labels = pd.read_csv(SCIN_LABELS_URL)
        cache.parent.mkdir(parents=True, exist_ok=True)
        labels.to_csv(cache, index=False)
    cases_cache = RAW_DIR / "scin_cases.csv"
    if cases_cache.exists():
        cases = pd.read_csv(cases_cache)
    else:
        cases = pd.read_csv(SCIN_CASES_URL)
        cases.to_csv(cases_cache, index=False)
    return labels, cases

def scin_candidates_for(condition_key: str, n: int = 30) -> list[dict]:
    """Return up to n SCIN cases matching condition_key, prioritized by
    Monk skin-tone diversity (tones 4-6 first to match VN demographics).
    """
    labels, cases = load_scin_metadata()
    # Check if image_1_path exists, fallback to image_path if the column name changed
    img_col = "image_1_path" if "image_1_path" in cases.columns else "image_path"
    
    labels["mapped_key"] = labels["dermatologist_skin_condition_on_label_name"].apply(map_label)
    matched = labels[labels["mapped_key"] == condition_key]

    # Join with cases for image_path + monk_skin_tone
    joined = matched.merge(cases, on="case_id", how="inner")

    # Priority sort: prefer Monk tones 4-6 (darker, matches VN), then any
    if "monk_skin_tone_label_us" in joined.columns:
        joined["_priority"] = joined["monk_skin_tone_label_us"].apply(
            lambda x: 0 if isinstance(x, str) and any(
                f"Monk {t}" in x for t in ["4", "5", "6"]
            ) else 1
        )
        joined = joined.sort_values("_priority")

    candidates = []
    for _, row in joined.head(n).iterrows():
        # SCIN image paths look like "scin_images/<case_id>_<n>.jpg"
        img_path = row.get(img_col)
        if not isinstance(img_path, str):
            continue
        url = SCIN_IMAGE_BASE + img_path.split("/")[-1]
        candidates.append({
            "url": url,
            "source_dataset": "SCIN",
            "license": "CC BY 4.0",
            "case_id": row["case_id"],
            "monk_tone": row.get("monk_skin_tone_label_us"),
            "fitzpatrick_type": None,
        })
    return candidates

# Cell — DermNet session-only candidates (no disk cache)
def dermnet_candidates_for(condition_key: str, n: int = 30) -> list[dict]:
    """Pull DermNet URLs from data/dataset_audit.json (TIP-001A output).
    Returns URL list; the describe pipeline fetches into memory only —
    never to disk. Per CC BY-NC-ND: we can display/describe but not
    redistribute the image bytes.
    """
    audit = json.loads(Path("data/dataset_audit.json").read_text())
    dermnet_urls = audit["datasets"]["dermnet_nz"]["image_urls_per_condition"]
    urls = dermnet_urls.get(condition_key, [])[:n]
    return [
        {
            "url": u,
            "source_dataset": "DermNet NZ",
            "license": "CC BY-NC-ND (session-only)",
            "case_id": None,
            "monk_tone": None,
            "fitzpatrick_type": None,
        }
        for u in urls
    ]

def candidates_for(condition_key: str, n: int = 30) -> list[dict]:
    """Per-condition source-aware candidate list.
    Returns up to n image candidates following SOURCE_POLICY.
    """
    sources = SOURCE_POLICY[condition_key]
    candidates = []
    remaining = n

    for source in sources:
        if remaining <= 0:
            break
        if source == "fitzpatrick17k":
            new = fitz_candidates_for(condition_key, remaining)
        elif source == "scin":
            new = scin_candidates_for(condition_key, remaining)
        elif source == "dermnet_session_only":
            new = dermnet_candidates_for(condition_key, remaining)
        else:
            raise ValueError(f"Unknown source: {source}")
        candidates.extend(new)
        remaining -= len(new)

    return candidates[:n]

DERMNET_HEADERS = {'User-Agent': USER_AGENT}

def fetch_image(candidate: dict) -> Image.Image | None:
    """Fetch image; for DermNet, never persist to disk."""
    url = candidate["url"]
    try:
        if candidate["source_dataset"] == "DermNet NZ":
            # Session-only: in-memory PIL Image, no disk cache
            import requests
            r = requests.get(url, headers=DERMNET_HEADERS, timeout=10)
            r.raise_for_status()
            return Image.open(BytesIO(r.content)).convert("RGB")
        else:
            # SCIN + Fitzpatrick17k: cache to data/raw/<source>_selected/
            import requests
            cache_dir = RAW_DIR / f"{candidate['source_dataset'].lower()}_selected"
            cache_dir.mkdir(parents=True, exist_ok=True)
            cache_path = cache_dir / f"{hash(url) & 0xFFFFFFFF:08x}.jpg"
            if cache_path.exists():
                return Image.open(cache_path).convert("RGB")
            r = requests.get(url, headers=DERMNET_HEADERS, timeout=10)
            r.raise_for_status()
            img = Image.open(BytesIO(r.content)).convert("RGB")
            img.save(cache_path, "JPEG", quality=92)
            time.sleep(0.5)  # polite
            return img
    except Exception as e:
        print(f"  ⚠ fetch failed: {url[:60]}... — {e}")
        return None

selected: dict[str, list[dict]] = {}
fetched_images: dict[str, list[Image.Image]] = {}

for key in CONDITIONS:
    cands = candidates_for(key, n=N_CANDIDATES)
    kept_cands = []
    kept_imgs = []
    print(f"\n{key}: {len(cands)} candidates available")
    for c in cands:
        if len(kept_cands) >= N_PER_CONDITION:
            break
        img = fetch_image(c)
        if img is not None:
            kept_cands.append(c)
            kept_imgs.append(img)
    selected[key] = kept_cands
    fetched_images[key] = kept_imgs
    print(f"  → fetched {len(kept_cands)}/{N_PER_CONDITION} successfully")

print('\n=== Selection summary ===')
for k in CONDITIONS:
    by_src = Counter(s['source_dataset'] for s in selected[k])
    print(f'  {k:25s} {len(selected[k])}/{N_PER_CONDITION}  ({dict(by_src)})')

## Cell 6 — Vietnamese describe prompt + sampling params

Deterministic settings: `temperature=0.0`, `seed=42`. The prompt
explicitly forbids diagnostic terms — descriptions must be
observational only. The `validation` step in Cell 8 catches any
leakage.

In [ ]:
DESCRIBE_PROMPT_VI = """Bạn là người hỗ trợ mô tả ảnh, KHÔNG phải bác sĩ.

Mô tả khách quan các đặc điểm hình ảnh bạn nhìn thấy trên ảnh tổn thương da:
- Màu sắc tổn thương
- Hình dạng và kích thước
- Bề mặt (vảy, mụn nước, loét, phẳng, gồ ghề...)
- Ranh giới với da lành (rõ hay mờ)
- Phân bố (tập trung, rải rác, đối xứng)

QUAN TRỌNG:
- CHỈ mô tả những gì thực sự nhìn thấy
- KHÔNG chẩn đoán
- KHÔNG đề cập tên bệnh
- Trả lời bằng tiếng Việt, 3-4 câu, ngắn gọn"""

PROMPT_VERSION = 'describe-v1.0'

## Cell 7 — Generate descriptions

vLLM's `llm.chat(...)` API accepts vision messages with PIL images.
The exact message shape for Qwen2.5-VL through vLLM is:

```python
[{
  'role': 'user',
  'content': [
    {'type': 'image', 'image': pil_image},
    {'type': 'text',  'text': prompt},
  ],
}]
```

If the installed vLLM version uses a different shape (e.g.,
`{'image_url': {'url': ...}}` for OpenAI-compatible APIs), adapt the
`describe_image()` body and document in the Completion Report.

In [ ]:
def describe_image(image: Image.Image) -> str:
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': DESCRIBE_PROMPT_VI},
        ],
    }]
    output = llm.chat(messages, sampling_params=QWEN_SAMPLING)
    return output[0].outputs[0].text.strip()


visual_db: dict[str, dict] = {}
peak_vram_gb = None
run_info_extras = {}

# Try to record peak VRAM usage
try:
    import torch
    torch.cuda.reset_peak_memory_stats()
except Exception:
    torch = None

# Cell — Determinism check (one-image, two-runs)
print("Running determinism check on first available image...")
first_cond = next(c for c in CONDITIONS if SOURCE_POLICY[c][0] != "dermnet_session_only")
test_candidates = candidates_for(first_cond, n=1)
if not test_candidates:
    print(f"⚠ No candidates for determinism check on {first_cond}; skipping")
else:
    test_img = fetch_image(test_candidates[0])
    if test_img is None:
        print(f"⚠ Could not fetch test image; skipping determinism check")
    else:
        d1 = describe_image(test_img)
        d2 = describe_image(test_img)
        if d1 == d2:
            print(f"✓ Determinism confirmed (Qwen + AWQ + temp=0 + seed=42 stable on T4)")
            run_info_extras["determinism_check"] = "pass"
        else:
            print(f"⚠ Determinism FAILED — outputs differ between identical runs.")
            print(f"  Run 1: {d1[:100]!r}...")
            print(f"  Run 2: {d2[:100]!r}...")
            print(f"  Continuing anyway, but visual_descriptions.json may not be byte-reproducible.")
            run_info_extras["determinism_check"] = "fail"
            run_info_extras["determinism_sample"] = {"run1": d1, "run2": d2}

for key in CONDITIONS:
    descriptions = []
    for i, (cand, img) in enumerate(zip(selected[key], fetched_images[key])):
        desc = describe_image(img)
        descriptions.append({
            'image_index': i,
            'source_dataset': cand["source_dataset"],
            'license': cand["license"],
            'case_id': cand.get("case_id"),
            'fitzpatrick_type': cand.get("fitzpatrick_type"),
            'monk_tone': cand.get("monk_tone"),
            'source_url': cand['url'],
            'description': desc,
        })
    visual_db[key] = {
        'name_vi': CONDITION_VN[key],
        'name_en': key.replace('_', ' ').title(),
        'n_descriptions': len(descriptions),
        'descriptions': descriptions,
    }
    print(f'✓ {CONDITION_VN[key]:30s} {len(descriptions)} descriptions')

# Cell — Coverage circuit-breaker
MIN_DESCRIPTIONS_PER_CONDITION = 3
failed_conditions = [
    cond for cond, info in visual_db.items()
    if info["n_descriptions"] < MIN_DESCRIPTIONS_PER_CONDITION
]
if failed_conditions:
    raise RuntimeError(
        f"Coverage circuit-breaker tripped. These conditions have "
        f"< {MIN_DESCRIPTIONS_PER_CONDITION} successful descriptions: "
        f"{failed_conditions}. "
        f"Investigate URL availability or augment sources before "
        f"shipping visual_descriptions.json. "
        f"Per-condition counts: "
        f"{ {c: i['n_descriptions'] for c, i in visual_db.items()} }"
    )
print(f"✓ Coverage check passed: all conditions have ≥ {MIN_DESCRIPTIONS_PER_CONDITION} descriptions")

if torch is not None and torch.cuda.is_available():
    peak_vram_gb = round(torch.cuda.max_memory_allocated() / 1024**3, 2)
    print(f'\nPeak VRAM during inference: {peak_vram_gb} GB')

## Cell 8 — Validate (forbidden terms check)

Per TIP-003 AC: descriptions must NOT mention diagnostic terms.
≤ 2 violations is acceptable; > 2 fails the AC and the Completion
Report must explain.

In [ ]:
# Quality check: descriptions should be 3-5 sentences
import re

def count_sentences(text: str) -> int:
    """Rough Vietnamese sentence count: split on . ! ? followed by space/end."""
    sentences = re.split(r"[.!?]+(?:\s|$)", text.strip())
    return sum(1 for s in sentences if s.strip())

flagged: list[dict] = []
short_descriptions = []
long_descriptions = []

for key, data in visual_db.items():
    for d in data['descriptions']:
        desc = d['description']
        
        # Original forbidden terms check
        text_lower = desc.lower()
        hits = [term for term in FORBIDDEN_TERMS if term in text_lower]
        if hits:
            flagged.append({
                'key': key,
                'image_index': d['image_index'],
                'forbidden_hits': hits,
                'description': desc,
            })
            
        # New sentence count check
        n_sentences = count_sentences(desc)
        if n_sentences < 3:
            short_descriptions.append({"key": key, "image_index": d["image_index"],
                                      "sentences": n_sentences,
                                      "description": desc})
        elif n_sentences > 6:
            long_descriptions.append({"key": key, "image_index": d["image_index"],
                                     "sentences": n_sentences})

if flagged:
    print(f'⚠ {len(flagged)} descriptions hit FORBIDDEN_TERMS — review:')
    for f in flagged[:10]:
        print(f'  [{f["key"]}/{f["image_index"]}] hits={f["forbidden_hits"]}: '
              f'{f["description"][:100]}...')
else:
    print('✓ No forbidden terms detected')

if short_descriptions:
    print(f"⚠ {len(short_descriptions)} descriptions are < 3 sentences — review")
if long_descriptions:
    print(f"⚠ {len(long_descriptions)} descriptions are > 6 sentences — review")

## Cell 9 — Write `data/visual_descriptions.json`

Schema matches TIP-003 spec exactly:

```
generated_at, model, model_settings, prompt_version,
source_dataset, source_dataset_license, n_conditions,
n_descriptions_per_condition, conditions, validation
```

`peak_vram_gb` and a per-source-dataset breakdown are added under
`run_info` for the Completion Report; they're additive to the spec
schema (do not remove any required field).

In [ ]:
out = {
    'generated_at': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    'model': QWEN_MODEL,
    'model_settings': {
        'temperature': QWEN_SAMPLING.temperature,
        'seed': QWEN_SAMPLING.seed,
        'max_tokens': QWEN_SAMPLING.max_tokens,
    },
    'prompt_version': PROMPT_VERSION,
    'source_datasets': [
        {"name": "Fitzpatrick17k", "license": "CC BY-NC-SA 4.0"},
        {"name": "SCIN", "license": "CC BY 4.0"},
        {"name": "DermNet NZ", "license": "CC BY-NC-ND (session-only)"},
    ],
    'source_dataset': 'multi-source',
    'source_dataset_license': (
        'CC BY-NC-SA 4.0' + (' / CC BY 4.0' if ENABLE_SCIN_FALLBACK else '')
    ),
    'n_conditions': len(CONDITIONS),
    'n_descriptions_per_condition': N_PER_CONDITION,
    'conditions': visual_db,
    'validation': {
        'diagnostic_term_violations': len(flagged),
        'flagged_items': flagged,
        'short_descriptions': short_descriptions,
        'long_descriptions': long_descriptions,
        'short_count': len(short_descriptions),
        'long_count': len(long_descriptions),
    },
    'run_info': {
        'peak_vram_gb': peak_vram_gb,
        'scin_fallback_enabled': ENABLE_SCIN_FALLBACK,
        'per_condition_source_breakdown': {
            k: dict(Counter(s['source_dataset'] for s in selected[k]))
            for k in CONDITIONS
        },
        **run_info_extras
    },
}

OUT_PATH = DATA_DIR / 'visual_descriptions.json'
OUT_PATH.write_text(
    json.dumps(out, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(f'✓ Wrote {OUT_PATH}: {OUT_PATH.stat().st_size:,} bytes')

print()
print('=' * 60)
print('TIP-003 — visual descriptions summary')
print('=' * 60)
for k in CONDITIONS:
    n = visual_db[k]['n_descriptions']
    flag = ' ⚠ < 5' if n < N_PER_CONDITION else ''
    print(f'  {k:25s} {n}/{N_PER_CONDITION}{flag}')
print()
print(f'Diagnostic term violations: {len(flagged)} '
      f'(AC: ≤ 2 with explanation)')
print(f'Peak VRAM: {peak_vram_gb} GB '
      f'(AC: < 13 GB on T4)')

## Cell 10 — Done

`data/visual_descriptions.json` is the committed artifact. Raw
images in `data/raw/fitz_selected/` are gitignored (license
attribution + non-redistribution).

**Determinism check (re-run produces identical output):**
re-run this notebook end-to-end. Confirm `visual_descriptions.json`
is byte-identical (modulo `generated_at`). If descriptions differ,
something else is non-deterministic — investigate before committing.

**Next:** TIP-005 will load `visual_descriptions.json` via
`backend/prompts/visual_context.py` (stub committed alongside this
notebook) and inject into the system prompt's `{visual_context}`
slot.